# ACTIVIDAD SESIÓN APACHE SPARK
Imagina que trabajas para una empresa de análisis de mercado y tu tarea es estudiar las preferencias
de los consumidores en Sudamérica en cuanto a los modelos de teléfonos inteligentes. Para esto,
se te ha entregado un conjunto de datos que contiene información sobre las marcas y modelos de
teléfonos más vendidos en distintos países de Sudamérica, junto con la edad de los compradores y
las características del teléfono (como la cantidad de memoria RAM, la capacidad de la batería, el
precio, etc.).

**INSTRUCCIONES**

1. **Instalación y configuración de PySpark (1 punto)**
- Configura correctamente el entorno de PySpark y crea una SparkSession con el nombre
AnalisisTelefonos.

In [ ]:
! pip install pyspark

In [ ]:
# Librerías
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count
from pyspark.sql import functions as F
from google.colab import files

In [ ]:
# Sesión
spark = SparkSession.builder\
    .appName("AnalisisTelefonos")\
    .master("local[*]") \
    .getOrCreate()

2. **Carga de datos (1 punto)**
- Carga el archivo CSV proporcionado (con el nombre telefonos_sudamerica.csv) en un
DataFrame de PySpark. Asegúrate de que el archivo contenga los encabezados.

In [ ]:
uploaded = files.upload()

Saving telefonos_sudamerica.csv to telefonos_sudamerica.csv


In [ ]:
df = spark.read.csv("telefonos_sudamerica.csv", header=True, inferSchema=True)
print(df.count())

10


3. **Exploración inicial de los datos (1 punto)**
- Muestra las primeras 10 filas del DataFrame y realiza una inspección básica de los tipos de
datos de cada columna. ¿Existen valores nulos o erróneos en alguna columna?

In [ ]:
# Primeras 10 filas
df.show(10, truncate=False)

+---------+-------------+--------+------+-----------+-----------------+--------------+-----------+
|pais     |modelo       |marca   |precio|memoria_ram|capacidad_bateria|edad_comprador|fecha_venta|
+---------+-------------+--------+------+-----------+-----------------+--------------+-----------+
|Argentina|Galaxy S21   |Samsung |1000  |8          |4000             |30            |2023-01-15 |
|Brasil   |iPhone 13    |Apple   |1200  |6          |3500             |25            |2023-02-10 |
|Chile    |Xiaomi Mi 11 |Xiaomi  |800   |6          |4500             |28            |2023-03-05 |
|Brasil   |Galaxy A72   |Samsung |700   |6          |5000             |22            |2023-02-12 |
|Peru     |Redmi Note 10|Xiaomi  |350   |4          |4000             |34            |2023-01-20 |
|Colombia |iPhone 12    |Apple   |950   |6          |3500             |27            |2023-02-18 |
|Brasil   |Motorola Edge|Motorola|750   |8          |5000             |31            |2023-01-25 |
|Argentina

In [ ]:
# Tipos de datos
df.printSchema()

root
 |-- pais: string (nullable = true)
 |-- modelo: string (nullable = true)
 |-- marca: string (nullable = true)
 |-- precio: integer (nullable = true)
 |-- memoria_ram: integer (nullable = true)
 |-- capacidad_bateria: integer (nullable = true)
 |-- edad_comprador: integer (nullable = true)
 |-- fecha_venta: date (nullable = true)



4. **Filtrado de datos (2 puntos)**
- Filtra el DataFrame para obtener únicamente los teléfonos vendidos en Brasil.
- Luego, filtra esos datos para obtener solo los teléfonos de la marca Samsung.

In [ ]:
# Filtro Brasil
df_brasil = df.filter(col("pais") == "Brasil")
print(df_brasil.count())

3


In [ ]:
# Filtro Samsung (en Brasil)
df_samsung = df_brasil.filter(col("marca") == "Samsung")
print(df_samsung.count())

1


5. **Operaciones de agrupamiento y agregación (2 puntos)**
- Agrupa los datos por país y calcula la venta promedio de los teléfonos (promedio de precio).
- Agrupa los datos por marca y calcula el número de teléfonos vendidos por cada marca.

In [ ]:
df.groupBy("pais").agg(avg("precio").alias("venta_promedio")).show()

+---------+-----------------+
|     pais|   venta_promedio|
+---------+-----------------+
|Argentina|            750.0|
|     Peru|            725.0|
|    Chile|            600.0|
|   Brasil|883.3333333333334|
| Colombia|            950.0|
+---------+-----------------+



In [ ]:
df.groupBy("marca").agg(count("*").alias("num_telefonos_vendidos")).show()

+--------+----------------------+
|   marca|num_telefonos_vendidos|
+--------+----------------------+
|Motorola|                     1|
|  Xiaomi|                     3|
| Samsung|                     4|
|   Apple|                     2|
+--------+----------------------+



6. **Análisis por rango de edad (1 punto)**
- Crea una nueva columna en el DataFrame que agrupe a los compradores por rango de edad
(por ejemplo, 18-25 años, 26-35 años, 36-50 años, 51+ años).
- Agrupa los datos por este nuevo rango de edad y muestra el promedio de precio de los
teléfonos vendidos para cada rango de edad.

In [ ]:
df.withColumn(
    "rango_edad",
    F.when((col("edad_comprador") >= 18) & (col("edad_comprador") <= 25), "18-25 años")
     .when ((col("edad_comprador") >= 26) & (col("edad_comprador") <= 35), "26-35 años")
     .when ((col("edad_comprador") >=36) & (col("edad_comprador") <= 50), "36-50 años")
     .otherwise("51+ años")
).groupBy("rango_edad").agg(avg("precio").alias("precio_promedio")).show()

+----------+-----------------+
|rango_edad|  precio_promedio|
+----------+-----------------+
|26-35 años|764.2857142857143|
|18-25 años|            800.0|
+----------+-----------------+



7. **Análisis de correlación (1 punto)**
- Calcula la correlación entre las columnas memoria_ram y precio. ¿Qué tipo de correlación
existe entre estas dos variables?

In [ ]:
corr = df.corr("memoria_ram", "precio")
print(f'Correlación columnas memoria_ram y precio: {corr:.2f}')

Correlación columnas memoria_ram y precio: 0.63


8. **Filtrado por características del teléfono (1 punto)**
- Filtra los teléfonos que tengan una memoria RAM mayor a 6 GB y una batería mayor a 4000
mAh. ¿Cuántos teléfonos cumplen con esta condición?

In [ ]:
df_filtrado = df.filter((col("memoria_ram") > 6) & (col("capacidad_bateria") > 4000))
print(f'Número de teléfonos con RAM > a 6 GB y batería > a 4000mAh: {df_filtrado.count()}')
df_filtrado.show()

Número de teléfonos con RAM > a 6 GB y batería > a 4000mAh: 2
+------+-------------+--------+------+-----------+-----------------+--------------+-----------+
|  pais|       modelo|   marca|precio|memoria_ram|capacidad_bateria|edad_comprador|fecha_venta|
+------+-------------+--------+------+-----------+-----------------+--------------+-----------+
|Brasil|Motorola Edge|Motorola|   750|          8|             5000|            31| 2023-01-25|
|  Peru|   Galaxy S21| Samsung|  1100|          8|             4500|            33| 2023-02-22|
+------+-------------+--------+------+-----------+-----------------+--------------+-----------+



9. **Guardar los resultados (1 punto)**
- Guarda el DataFrame resultante de los filtros y agregaciones anteriores en un nuevo archivo
CSV llamado resultados_analisis.csv.

In [ ]:
# Escribir como carpeta temporal con un solo 'part-*.csv'
out_dir = "/content/resultados_analisis"
df_filtrado.coalesce(1).write.mode("overwrite").option("header", True).csv(out_dir)
# coalesce(1): reduce a 1 partición, así Spark escribe un solo part-*.csv.
# Nota: coalesce(1) es costoso si el DF es grande (movimiento de datos). Úsalo solo si necesitas de verdad un único archivo.
# mode("overwrite"): borra la carpeta si ya existía.
# option("header", True): incluye encabezados en el CSV.
# .csv(out_dir): Spark siempre escribe una carpeta (out_dir) que contiene el(s) archivo(s) part-*.csv y metadatos _SUCCESS.

# Mover/renombrar al archivo final 'penguins_resultados.csv'
import os, glob, shutil
dest = "/content/resultado_analisis.csv"
# dest es el nombre final que quieres (un archivo, no carpeta).

# Limpiar si quedó una carpeta/archivo previo con el mismo nombre
if os.path.exists(dest):
    if os.path.isdir(dest):
        shutil.rmtree(dest)
    else:
        os.remove(dest)
# Evita el error IsADirectoryError que ya viste: si dest existía como carpeta de intentos anteriores, hay que borrarla de forma distinta que un archivo.

part = glob.glob(os.path.join(out_dir, "part-*.csv"))
if not part:
    raise FileNotFoundError("No se encontró part-*.csv en " + out_dir)
# Busca el único CSV que Spark generó dentro de out_dir.
# Si no existe, se lanza un error explicativo (descarga fallida, permisos, etc.).

shutil.move(part[0], dest)
shutil.rmtree(out_dir, ignore_errors=True)
# Mueve/renombra el part-*.csv al nombre final dest.
# Elimina la carpeta temporal out_dir para dejar todo limpio.

print("✅ Archivo final listo en:", dest)
# Mensaje final con la ruta del CSV único.

✅ Archivo final listo en: /content/resultado_analisis.csv


In [ ]:
files.download(dest)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>